# LangGraph G3 — Tools and the agent loop
CampusAI v1 can chat but cannot look anything up. Give it **tools**: Python functions the model
may *request*. The model never runs them; a node of ours does. That is the whole agent loop,
and in LangGraph it is one conditional edge pointing backwards:

```text
              +---------+   tool calls?   +---------+
  START --->  |  agent  | ------yes-----> |  tools  |
              +---------+                 +---------+
                   | no                        |
                   v                           |
                  END   <----------------------+   (back to agent with the results)
```

This shape is called **ReAct** (reason, act, observe, repeat). Every "agent" you will meet in
industry is this loop with layers around it. We build it by hand once so that nothing about it
is mysterious, then meet LangGraph's prebuilt helpers, then learn what makes a tool *good*.

### Step 1 — Tools, and what the model sees

The `@tool` decorator comes from LangChain: it turns a function into an object with a name, a
description (the docstring) and an argument schema built from the type hints. That JSON is
sent to the model with every request, so **the docstring is the interface**.

In [ ]:
from langchain.tools import tool                       # LangChain: turns a function into a tool the model can request

@tool                                                  # LangChain decorator; the function body is ours
def get_student(student_id: str) -> str:
    """Look up a student record by id such as 'S001'. Returns name, programme, year, attendance %, credits and email."""
    record = STUDENTS.get(student_id)
    return json.dumps({"id": student_id, **record} if record else {"error": "student_not_found", "student_id": student_id})

@tool
def get_course(course_code: str) -> str:
    """Look up a course by code such as 'CS201'. Returns title, credits, seats left, prerequisites and description."""
    record = COURSES.get(course_code)
    return json.dumps({"code": course_code, **record} if record else {"error": "course_not_found", "course_code": course_code})

@tool
def search_handbook(query: str) -> str:
    """Search the university handbook (attendance, credits, retakes, registration, fees). Returns the two most relevant paragraphs."""
    words = {w for w in re.findall(r"[a-z]+", query.lower()) if len(w) > 3}          # ignore short filler words
    def score(item):
        topic, text = item
        overlap = len(words & set(re.findall(r"[a-z]+", text.lower())))
        return -(overlap + (10 if topic in words or topic.rstrip("s") in words else 0))   # the topic word itself counts most
    ranked = sorted(HANDBOOK.items(), key=score)
    return "\n".join(f"[{topic}] {text}" for topic, text in ranked[:2])

READ_TOOLS = [get_student, get_course, search_handbook]
print("tool the model sees:", json.dumps(convert_to_openai_tool(get_student)["function"], indent=1)[:300], "...")   # LangChain: the JSON schema

### Step 2 — The agent node, the tools node and the routing function, all ours

Three small functions. `call_model` binds the tools and asks the model. `run_tools` executes
every requested call and returns the results as `ToolMessage`s. `should_continue` looks at the
last message: tool calls present means go to *tools*, otherwise finish.

In [ ]:
def call_model(state: ChatState):                                  # ours: the agent node
    llm = model.bind_tools(READ_TOOLS)                             # LangChain: attach the tool schemas to the request
    reply = llm.invoke([SystemMessage(CAMPUS_PERSONA + " Use tools for any student, course or handbook fact.")] + state["messages"])   # LangChain
    return {"messages": [reply]}

def run_tools(state: ChatState):                                   # ours: the tools node
    by_name = {t.name: t for t in READ_TOOLS}                      # t.name is a LangChain tool attribute
    last = state["messages"][-1]
    results = [by_name[call["name"]].invoke(call) for call in last.tool_calls]   # LangChain: tool.invoke(tool_call) -> ToolMessage
    return {"messages": results}

def should_continue(state: ChatState):                             # ours: routing function
    return "tools" if state["messages"][-1].tool_calls else END     # last.tool_calls is a LangChain attribute

g = StateGraph(ChatState)
g.add_node("agent", call_model)                                    # LangGraph
g.add_node("tools", run_tools)
g.add_edge(START, "agent")
g.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})   # LangGraph: the loop's decision point
g.add_edge("tools", "agent")                                       # LangGraph: back to the model with the results
campusai_v2 = g.compile()

out = campusai_v2.invoke({"messages": [HumanMessage("What is student S001's attendance, and what does the handbook say about attendance?")]})
show_messages(out["messages"])

### Step 3 — The same graph with LangGraph's prebuilt pieces

`ToolNode` is a ready-made tools node and `tools_condition` a ready-made `should_continue`.
Use them from now on, knowing exactly what they do. `build_agent()` wraps the pattern so later
sections can create agents in one line. Both graphs draw identically.

In [ ]:
from langgraph.prebuilt import ToolNode, tools_condition   # LangGraph: prebuilt tools node and routing function

def build_agent(tools, persona=CAMPUS_PERSONA, checkpointer=None, store=None, context_schema=None):   # ours: the loop as a reusable function
    def agent_node(state: ChatState):
        reply = model.bind_tools(tools).invoke([SystemMessage(persona + " Use tools for any student, course or handbook fact.")] + state["messages"])   # LangChain
        return {"messages": [reply]}
    g = StateGraph(ChatState, context_schema=context_schema)
    g.add_node("agent", agent_node)                             # ours
    g.add_node("tools", ToolNode(tools))                        # LangGraph: replaces our run_tools
    g.add_edge(START, "agent")
    g.add_conditional_edges("agent", tools_condition)           # LangGraph: replaces our should_continue (routes to "tools" or END)
    g.add_edge("tools", "agent")
    return g.compile(checkpointer=checkpointer, store=store)   # LangGraph

campusai_v2b = build_agent(READ_TOOLS)
out = campusai_v2b.invoke({"messages": [HumanMessage("Does CS201 have seats left, and what are its prerequisites?")]})
show_messages(out["messages"])
print(campusai_v2b.get_graph().draw_mermaid())              # LangGraph: agent <-> tools, exactly the diagram above

### Step 4 — Tool design: the description is the interface, arguments are validated, errors are data

Three habits separate a demo tool from a production tool. **Describe precisely**: the model
chooses tools by reading descriptions, so a vague one is a bug. **Validate arguments** with a
Pydantic schema so bad input never reaches your systems. **Return errors as values** (a dict
with an `error` key) instead of raising, so the model can read what went wrong and recover.

In [ ]:
from pydantic import BaseModel, Field, field_validator        # Pydantic: validation library that LangChain uses for schemas

@tool
def lookup(id: str) -> str:
    """Lookup something."""
    return json.dumps(STUDENTS.get(id, {"error": "not found"}))

for label, tools in [("vague 'lookup'   ", [lookup]), ("precise get_student", [get_student])]:
    out = build_agent(tools).invoke({"messages": [HumanMessage("What programme is student S002 on?")]})
    used = [c["name"] for m in out["messages"] if isinstance(m, AIMessage) for c in m.tool_calls]
    print(f"{label} -> tools used: {str(used or 'none'):18} answer: {text_of(out['messages'][-1])[:70]}")

class StudentLookup(BaseModel):                                # ours: the argument schema
    student_id: str = Field(description="Student id in the form 'S' followed by three digits, e.g. 'S001'.")
    @field_validator("student_id")
    @classmethod
    def check_format(cls, value):
        if not re.fullmatch(r"S\d{3}", value):
            raise ValueError("student_id must look like S001")
        return value

@tool(args_schema=StudentLookup)                               # LangChain: validate arguments with our Pydantic schema
def get_student(student_id: str) -> str:
    """Look up a student record by id such as 'S001'. Returns name, programme, year, attendance %, credits and email."""
    record = STUDENTS.get(student_id)
    return json.dumps({"id": student_id, **record} if record else {"error": "student_not_found", "student_id": student_id})   # error as data

READ_TOOLS = [get_student, get_course, search_handbook]
print("\nvalid   :", get_student.invoke({"student_id": "S003"})[:60])
print("missing :", get_student.invoke({"student_id": "S999"}))
try:
    get_student.invoke({"student_id": "drop table students"})
except Exception as exc:
    print("invalid :", type(exc).__name__, "- rejected before any code ran")

### Recap

- **Problem seen:** the chatbot could not look anything up, and a badly described tool is never chosen.
- **Layer added:** LangChain tools, an agent node, a tools node, a conditional edge back to the agent; `ToolNode`, `tools_condition`, `build_agent()`; validated, error-returning tools.
- **Evidence:** the trajectory shows requests, results and the answer; the vague tool went unused; the bad id never reached the data.